## Architecture & Execution Context Cheat Sheet

Before we dive into code, understand this critical fact:

**Your notebook runs a LOCAL PyFlink mini-cluster in the Jupyter kernel.**
- When you call `env.execute_async()`, it starts a small Flink JVM inside your Python process
- This is DIFFERENT from the Docker Flink cluster (port 8081)
- Your job will NOT appear in http://localhost:8081 (that's for Docker-submitted jobs)
- Both execution contexts can coexist; they communicate via shared Kafka and Cassandra

Think of it this way:
```
Notebook Mini-Cluster (Your Job)          Docker Flink UI (Separate)
         ↓                                         ↓
    Reads raw.gps         ←← Kafka Bus ←→      Displays Docker jobs
    Writes processed.gps
    Writes to Cassandra   ←← Cassandra ←→      Displays cluster metrics
         ↓
    Job runs asynchronously (non-blocking)
    You remain in notebook to write more code
```

**How to verify your job is working:**
- ❌ NOT in Flink UI (expected!)
- ✅ Check Kafka: `docker exec taasim-kafka ... --topic processed.gps`
- ✅ Check Cassandra: `docker exec taasim-cassandra cqlsh -e "SELECT * FROM taasim.vehicle_positions"`
- ✅ Check Grafana: Panels query Cassandra and display results

Key variables you'll use:
- `env`: StreamExecutionEnvironment (the Flink context)
- `job_client`: Returned from execute_async(), allows monitoring (get_job_id(), get_job_status(), etc.)
- `*_stream`: DataStreams (transformable pipelines of data)

# Advanced Big Data Capstone: TaaSim Urban Mobility
## Week 3: Stream Processing I - GPS Normalizer (Flink Job 1)

**Objective:** Ingest raw GPS streams from Kafka, assign event-time watermarks (3-min allowed lateness), validate coordinates, map-match to Casablanca zones, anonymize data, and sink to Cassandra and Kafka.

## Step 1: Hook the Schema into your Kafka Consumer
Your Flink job needs to read the raw.gps topic. You will use the JSON schema you just wrote inside a FlinkKafkaConsumer to deserialize the raw byte stream into structured objects (like Java POJOs, Scala case classes, or PyFlink Rows).

Action: Instantiate your Kafka source and pass your JSON deserializer to it. This transforms the Kafka stream into a Flink DataStream.

In [50]:
import os, shutil, urllib.request, sys
from pathlib import Path
from pyflink.find_flink_home import _find_flink_home
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.connectors.kafka import FlinkKafkaConsumer
from pyflink.common.serialization import SimpleStringSchema
from pyflink.common.typeinfo import Types
import json

# Toggle this to True only for short debugging sessions.
DEBUG_STREAM_OUTPUT = False

# Fix Java on Windows.
if os.name == "nt":
    for home in [
        r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot",
        r"C:\Program Files\Adoptium\jdk-17.0.7",
    ]:
        java_bin = Path(home) / "bin" / "java.exe"
        if java_bin.exists():
            os.environ["JAVA_HOME"] = str(Path(home))
            os.environ["PATH"] = f"{java_bin.parent}{os.pathsep}" + os.environ.get("PATH", "")
            break

java_cmd = shutil.which("java")
if not java_cmd:
    raise RuntimeError("Java not found. Install JDK 11/17.")
print(f"Using Java: {java_cmd}")

# Required jars: Kafka + Cassandra connectors and Kafka client dependency.
flink_lib = Path(_find_flink_home()) / "lib"
jars = {
    "flink-connector-kafka-3.0.2-1.18.jar":
        "https://repo1.maven.org/maven2/org/apache/flink/flink-connector-kafka/3.0.2-1.18/flink-connector-kafka-3.0.2-1.18.jar",
    "kafka-clients-3.2.3.jar":
        "https://repo1.maven.org/maven2/org/apache/kafka/kafka-clients/3.2.3/kafka-clients-3.2.3.jar",
    "flink-connector-cassandra_2.12-3.2.0-1.18.jar":
        "https://repo1.maven.org/maven2/org/apache/flink/flink-connector-cassandra_2.12/3.2.0-1.18/flink-connector-cassandra_2.12-3.2.0-1.18.jar",
}

for jar_name, url in jars.items():
    dest = flink_lib / jar_name
    if not dest.exists():
        print(f"Downloading {jar_name} ...")
        urllib.request.urlretrieve(url, dest)
        print(f"  -> Saved to {dest}")
    else:
        print(f"  OK: {jar_name} already present")

# Force Flink Python operators to use the same interpreter as this Jupyter kernel.
os.environ["PYFLINK_CLIENT_EXECUTABLE"] = sys.executable
os.environ["PYFLINK_PYTHON_EXECUTABLE"] = sys.executable

env = StreamExecutionEnvironment.get_execution_environment()
env.set_python_executable(sys.executable)
env.set_parallelism(1)
print(f"PyFlink worker python: {sys.executable}")
print("Flink environment initialized.")

Using Java: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot\bin\java.EXE
  OK: flink-connector-kafka-3.0.2-1.18.jar already present
  OK: kafka-clients-3.2.3.jar already present
  OK: flink-connector-cassandra_2.12-3.2.0-1.18.jar already present
PyFlink worker python: c:\Users\nmira\AppData\Local\Programs\Python\Python311\python.exe
Flink environment initialized.


### Step 0: Environment Setup - Why JARs, Java Discovery, and Python Pinning Matter

**The Problem**: PyFlink needs Java (to compile Flink operators) and connector libraries (JARs) to talk to Kafka and Cassandra. By default, these might not be available in your Python environment.

**The Solution**: This cell automates discovery and download.

**What Each Part Does**:

1. **Java Discovery** (Windows special case):
   - Searches for JDK 17 in standard Windows install paths
   - Sets JAVA_HOME and PATH environment variables
   - Why? PyFlink calls `javac` at runtime to compile your transformations

2. **JAR Download** (Maven Central):
   - `flink-connector-kafka`: Teaches Flink how to read from Kafka topics
   - `kafka-clients`: Kafka client library (protocol implementation)
   - `flink-connector-cassandra`: Teaches Flink how to write to Cassandra
   - Why download? These connectors aren't bundled with PyFlink; you add them optionally

3. **Python Executable Pinning**:
   - Sets PYFLINK_PYTHON_EXECUTABLE to your current Jupyter kernel
   - Why? Flink spawns worker processes; without this, they might use wrong Python

4. **StreamExecutionEnvironment**:
   - `env = StreamExecutionEnvironment.get_execution_environment()`
   - This creates the Flink mini-cluster context in your notebook
   - All transformations will be built on top of this `env` object
   - `set_parallelism(1)` ensures single-threaded processing (simpler debugging in notebook)

**Data Flow After This Cell**:
```
Your Machine
├─ JDK ✓ (found or error)
├─ JARs ✓ (downloaded to Flink lib/)
├─ Python Worker ✓ (pinned to Jupyter kernel)
└─ env object ✓ (ready to build job graph)
```

In [51]:
# Cell 2: Source & Schema
kafka_props = {
    'bootstrap.servers': 'localhost:9094', #extrernal port mapped to kafka broker
    'group.id': 'taasim-jupyter-debugger', #grouping in consumer helps load balancing(fault tolerance and parallelism)
    'auto.offset.reset': 'earliest' #start consuming from earliest messages (all historical data)
}

kafka_source = FlinkKafkaConsumer(
    topics='raw.gps',
    deserialization_schema=SimpleStringSchema(), #will handle deserialization manually in next step
    properties=kafka_props
)
raw_stream = env.add_source(kafka_source) #creates data stream from kafka source

def parse_gps_json(data_string): 
    try:
        data = json.loads(data_string)
        return (
            int(data['taxi_id']),
            int(data['timestamp']),
            float(data['latitude']),
            float(data['longitude']),
            float(data['speed']),
            str(data['status'])
        )
    except Exception:
        return None
#structed stream with schema 
structured_stream = raw_stream \
    .map(
        parse_gps_json,
        output_type=Types.TUPLE([
            Types.INT(), Types.LONG(), Types.FLOAT(),
            Types.FLOAT(), Types.FLOAT(), Types.STRING()
        ])
    ) \
    .filter(lambda x: x is not None)

print("✅ Data source and schema mapped.")


✅ Data source and schema mapped.


### Step 1: Source Configuration - Reading from Kafka raw.gps

**The Goal**: Connect to Kafka, consume JSON GPS events, parse them into structured tuples.

**Key Concepts**:

1. **Kafka Properties**:
   - `bootstrap.servers: 'localhost:9094'`: The advertised listener for external (non-Docker) clients like your notebook
   - `group.id`: Consumer group (Kafka tracks offsets per group; useful for scaling)
   - `auto.offset.reset: 'latest'`: Start from NEW messages only (not all history from topic creation)
   - Why localhost:9094? Because you're running Jupyter on Windows, not in Docker

2. **FlinkKafkaConsumer**:
   - Subscribes to 'raw.gps' topic
   - Deserializes bytes as JSON strings (SimpleStringSchema)
   - Returns `raw_stream`: a DataStream of strings
   - Data contract: Each string is JSON with structure `{"taxi_id": int, "timestamp": int, "latitude": float, "longitude": float, "speed": float, "status": string}`

3. **Parsing Function** (`parse_gps_json`):
   - Input: JSON string (bytes decoded)
   - Logic: json.loads() → dictionary → extract 6 fields, cast to correct types
   - Output: Tuple6 (taxi_id, timestamp, lat, lon, speed, status) OR None (if parse fails)
   - Error handling: Exceptions caught, return None → filtered out next

4. **Type System**:
   - Flink is strongly typed; each stream has a type
   - `raw_stream`: DataStream<String>
   - `structured_stream`: DataStream<Tuple6<Int, Long, Float, Float, Float, String>>
   - Why types? Enables optimization, schema enforcement, serialization efficiency

**Data Flow**:
```
Kafka raw.gps (JSON bytes)
        │
        ↓ FlinkKafkaConsumer (deserialize to string)
        │
raw_stream: "{"taxi_id":123,"timestamp":1700001234,...}"
        │
        ↓ map(parse_gps_json)
        │
 (123, 1700001234, 33.6, -7.64, 42.5, "AVAILABLE")  ← structured, typed, ready for transformation
        │
        ↓ filter(lambda x: x is not None)
        │
structured_stream: Only valid tuples (malformed dropped)
```

### Step 1: Source Configuration - Reading from Kafka raw.gps

**The Goal**: Connect to Kafka, consume JSON GPS events, parse them into structured tuples.

**Key Concepts**:

1. **Kafka Properties**:
   - `bootstrap.servers: 'localhost:9094'`: The advertised listener for external (non-Docker) clients like your notebook
   - `group.id`: Consumer group (Kafka tracks offsets per group; useful for scaling)
   - `auto.offset.reset: 'latest'`: Start from NEW messages only (not all history from topic creation)
   - Why localhost:9094? Because you're running Jupyter on Windows, not in Docker

2. **FlinkKafkaConsumer**:
   - Subscribes to 'raw.gps' topic
   - Deserializes bytes as JSON strings (SimpleStringSchema)
   - Returns `raw_stream`: a DataStream of strings
   - Data contract: Each string is JSON with structure `{"taxi_id": int, "timestamp": int, "latitude": float, "longitude": float, "speed": float, "status": string}`

3. **Parsing Function** (`parse_gps_json`):
   - Input: JSON string (bytes decoded)
   - Logic: json.loads() → dictionary → extract 6 fields, cast to correct types
   - Output: Tuple6 (taxi_id, timestamp, lat, lon, speed, status) OR None (if parse fails)
   - Error handling: Exceptions caught, return None → filtered out next

4. **Type System**:
   - Flink is strongly typed; each stream has a type
   - `raw_stream`: DataStream<String>
   - `structured_stream`: DataStream<Tuple6<Int, Long, Float, Float, Float, String>>
   - Why types? Enables optimization, schema enforcement, serialization efficiency

**Data Flow**:
```
Kafka raw.gps (JSON bytes)
        │
        ↓ FlinkKafkaConsumer (deserialize to string)
        │
raw_stream: "{"taxi_id":123,"timestamp":1700001234,...}"
        │
        ↓ map(parse_gps_json)
        │
 (123, 1700001234, 33.6, -7.64, 42.5, "AVAILABLE")  ← structured, typed, ready for transformation
        │
        ↓ filter(lambda x: x is not None)
        │
structured_stream: Only valid tuples (malformed dropped)
```

In [52]:
# Cell 3: Watermark Strategy
from pyflink.common.watermark_strategy import WatermarkStrategy, TimestampAssigner
from pyflink.common.time import Duration

class GPSTimestampAssigner(TimestampAssigner):
    def extract_timestamp(self, value, record_timestamp):
       #extract GPS timestamp and convert it to milliseconds because fo flink requiremenets (millisceoncds)
        return int(value[1] * 1000)

watermark_strategy = WatermarkStrategy \
        .for_bounded_out_of_orderness(Duration.of_minutes(3)) \
        .with_timestamp_assigner(GPSTimestampAssigner())
#apply strategy to structed stream to get event time with watermarks
event_time_stream = structured_stream.assign_timestamps_and_watermarks(watermark_strategy)

if DEBUG_STREAM_OUTPUT:
    event_time_stream.print()
    print("Debug sink attached for event_time_stream.")
else:
    print("Debug sink disabled for event_time_stream.")

print("✅ Watermarks assigned.")

Debug sink disabled for event_time_stream.
✅ Watermarks assigned.


### Step 2: Event-Time & Watermarks - Why We Don't Trust Processing Time

**The Core Problem with Processing Time**:
Imagine a taxi loses signal, buffers 10 GPS pings, then suddenly transmits all 10 at once.
- **Processing Time Logic**: "These 10 events arrived at exactly 14:30, so demand spiked at 14:30!"
- **Reality**: The events occurred spread over 5 minutes (14:20-14:25), but arrived as a burst
- **Business Impact**: Your dashboard would show a false demand spike, misleading operations

**The Solution**: Event Time + Watermarks

1. **Event Time**: The TRUE moment the GPS event occurred (taxi's clock)
   - Extracted from the `timestamp` field in each record
   - Represents reality, not network delays

2. **Timestamp Assigner**:
   - Custom logic: `extract_timestamp(value) → value[1] * 1000`
   - Takes field[1] (timestamp in SECONDS from raw data)
   - Multiplies by 1000 (Flink uses milliseconds internally)
   - Returns millisecond-precision epoch timestamp

3. **Watermark Strategy**:
   - `BoundedOutOfOrdernessWatermarks(Duration.of_minutes(3))`
   - Meaning: "After no new timestamps arrive for 3 minutes, finalize all open windows"
   - Ensures late-arriving buffered GPS pings (within 3 min lateness) are still counted
   - Handles out-of-order delivery automatically

**Watermark Flow Example**:
```
Event 1: taxi_id=123, timestamp=1700001200s (14:00:00)  arrives at Flink at 14:00:05
Event 2: taxi_id=456, timestamp=1700001205s (14:00:05)  arrives at Flink at 14:00:10
Event 3: taxi_id=789, timestamp=1700001210s (14:00:10)  arrives at Flink at 14:00:15

Watermark advancement: Now=14:00:15, so Watermark=14:00:12 (15min - 3min buffer)

Timeline: Any events with timestamp < 14:00:12 are finalized
          Any events with > 14:00:12 still might arrive within 3min buffer

Event 4 (late): taxi_id=123, timestamp=1700001207s (14:00:07) arrives at 14:00:18
                Event time 14:00:07 > watermark 14:00:12? NO → still within window, counted ✓
```

**Result**: `event_time_stream` carries both data AND implicit watermark signals → ready for windowed transformations

### Step 2: Taming the Chaos with Watermarks ⏱️

This step implements **Watermarks**, the mechanism that separates amateur stream processing from production-grade data engineering. 

#### The "Why": Handling Casablanca's Urban Reality
In stream processing, we must explicitly define how our system understands time. There are two distinct concepts:
1. **Processing Time:** The exact moment our Flink server receives the data packet.
2. **Event Time:** The actual moment the event occurred in the real world (e.g., when the taxi's GPS sensor actually recorded the coordinate).

**The Problem (Out-of-Order Events):** Imagine a *petit taxi* driving through the underpass near the Hassan II Mosque. It loses its 4G connection for two minutes. When it emerges, it suddenly transmits a batch of buffered GPS pings all at once. If our pipeline relies strictly on *Processing Time*, Flink will process all those pings as if they happened at the exact same second, creating a fake, concentrated demand spike on our dashboard.

**The Solution:** We enforce **Event Time** processing using a **Watermark Strategy**. By assigning a `BoundedOutOfOrdernessWatermarks` with a exactly 3-minute maximum allowed lateness, we give Flink a set of rules: *"When a 30-second time window closes, do not finalize the aggregations immediately. Wait up to 3 minutes for any lagging taxis to report their delayed pings. After 3 minutes, close the window permanently and drop anything older."*

### Step 3: Data Validation and Filtering

Before spatial mapping, filter corrupted GPS events so downstream metrics stay trustworthy.

Validation rules:
1. Speed limit: drop pings with speed > 150 km/h.
2. Geofencing: keep only coordinates inside the Casablanca bounding box.

In [53]:
if "job_client" in globals():
    try:
        job_id = job_client.get_job_id()
        status = job_client.get_job_status().result()
        print(f"JobID: {job_id}")
        print(f"Job status: {status}")
        print("Job runs asynchronously in the notebook kernel.")
    except Exception as ex:
        print(f"Could not read job status: {ex}")
else:
    print("No job submitted yet. Run the final execution cell first.")

JobID: 9509ba3ababde326f0b329a40b1295c5
Job status: JobStatus.RUNNING
Job runs asynchronously in the notebook kernel.


In [54]:
def is_valid_gps(record):
    latitude = record[2]
    longitude = record[3]
    speed = record[4]

    if speed > 150.0:
        return False

    valid_lat = 33.4 <= latitude <= 33.7
    valid_lon = -7.8 <= longitude <= -7.4
    return valid_lat and valid_lon

valid_stream = event_time_stream.filter(is_valid_gps)
if DEBUG_STREAM_OUTPUT:
    valid_stream.print()
    print("Debug sink attached for valid_stream.")
else:
    print("Debug sink disabled for valid_stream.")

print("Validation filters configured: speed <= 150 and Casablanca bounding box.")

Debug sink disabled for valid_stream.
Validation filters configured: speed <= 150 and Casablanca bounding box.


### Step 3: Data Quality Filters - Removing Garbage Before It Spreads

**Why Validation First?** 
Bad data downstream (in analytics, dashboards, databases) is worse than no data. The earlier you filter, the less waste propagates through your pipeline.

**Validation Rules**:

1. **Speed Limit** (≤ 150 km/h):
   - Taxi maximum is ~120 km/h
   - Anything faster likely indicates:
     - GPS sensor malfunction (jumped to random coordinate)
     - Data corruption during transmission
     - Vehicle not a taxi (ambulance/police → outside scope)

2. **Geofencing** (Casablanca bounding box):
   - Valid latitude: [33.4°, 33.7°] (covers Casablanca city)
   - Valid longitude: [-7.8°, -7.4°] (covers Casablanca city)
   - Why? Only care about Casablanca market; outside = out-of-service, roaming, errors

**Filter Logic**:
```
For each record (taxi_id, event_time, lat, lon, speed, status):
  IF speed > 150.0:
    → DROP (invalid)
  IF NOT (33.4 ≤ lat ≤ 33.7):
    → DROP (outside Casablanca)
  IF NOT (-7.8 ≤ lon ≤ -7.4):
    → DROP (outside Casablanca)
  ELSE:
    → PASS (valid, continue downstream)

Result: valid_stream contains only records that passed ALL checks
```

**Impact on Pipeline**:
- Input: 100 events/second from raw.gps
- Expected output: ~80-85 events/second (10-15% of raw data is typically noisy)
- Invalid events: silently dropped, not logged (normal for stream processing)

**Next Step**: Only valid events proceed to map-matching and anonymization

### Step 4: Geospatial Map-Matching and Anonymization

Objective: map each valid GPS ping to a Casablanca zone and replace raw coordinates with that zone centroid.

Cassandra target schema reminder: city, zone_id, event_time, taxi_id, latitude, longitude, speed, status.

In [55]:
CASABLANCA_ZONES = {
    1: {"name": "Anfa", "min_lat": 33.58, "max_lat": 33.62, "min_lon": -7.66, "max_lon": -7.62, "centroid_lat": 33.60, "centroid_lon": -7.64},
    2: {"name": "Sidi Belyout", "min_lat": 33.58, "max_lat": 33.61, "min_lon": -7.62, "max_lon": -7.58, "centroid_lat": 33.595, "centroid_lon": -7.60},
    3: {"name": "Roches Noires", "min_lat": 33.60, "max_lat": 33.64, "min_lon": -7.58, "max_lon": -7.54, "centroid_lat": 33.62, "centroid_lon": -7.56},
    4: {"name": "Ain Sebaa", "min_lat": 33.64, "max_lat": 33.70, "min_lon": -7.54, "max_lon": -7.48, "centroid_lat": 33.67, "centroid_lon": -7.51},
    5: {"name": "Hay Mohammadi", "min_lat": 33.60, "max_lat": 33.65, "min_lon": -7.56, "max_lon": -7.50, "centroid_lat": 33.625, "centroid_lon": -7.53},
    6: {"name": "Mers Sultan", "min_lat": 33.57, "max_lat": 33.60, "min_lon": -7.62, "max_lon": -7.58, "centroid_lat": 33.585, "centroid_lon": -7.60},
    7: {"name": "Maarif", "min_lat": 33.57, "max_lat": 33.60, "min_lon": -7.64, "max_lon": -7.60, "centroid_lat": 33.585, "centroid_lon": -7.62},
    8: {"name": "Sidi Othmane", "min_lat": 33.55, "max_lat": 33.59, "min_lon": -7.58, "max_lon": -7.52, "centroid_lat": 33.57, "centroid_lon": -7.55},
    9: {"name": "Sbata", "min_lat": 33.56, "max_lat": 33.59, "min_lon": -7.56, "max_lon": -7.50, "centroid_lat": 33.575, "centroid_lon": -7.53},
    10: {"name": "Ben M'Sick", "min_lat": 33.55, "max_lat": 33.58, "min_lon": -7.52, "max_lon": -7.46, "centroid_lat": 33.565, "centroid_lon": -7.49},
    11: {"name": "Ain Chock", "min_lat": 33.54, "max_lat": 33.58, "min_lon": -7.60, "max_lon": -7.54, "centroid_lat": 33.56, "centroid_lon": -7.57},
    12: {"name": "Hay Hassani", "min_lat": 33.54, "max_lat": 33.58, "min_lon": -7.66, "max_lon": -7.60, "centroid_lat": 33.56, "centroid_lon": -7.63},
    13: {"name": "Sidi Abderrahmane", "min_lat": 33.56, "max_lat": 33.60, "min_lon": -7.70, "max_lon": -7.66, "centroid_lat": 33.58, "centroid_lon": -7.68},
    14: {"name": "Ain Diab", "min_lat": 33.58, "max_lat": 33.62, "min_lon": -7.70, "max_lon": -7.66, "centroid_lat": 33.60, "centroid_lon": -7.68},
    15: {"name": "Dar Bouazza", "min_lat": 33.52, "max_lat": 33.58, "min_lon": -7.74, "max_lon": -7.66, "centroid_lat": 33.55, "centroid_lon": -7.70},
    16: {"name": "Sidi Moumen", "min_lat": 33.62, "max_lat": 33.68, "min_lon": -7.52, "max_lon": -7.46, "centroid_lat": 33.65, "centroid_lon": -7.49},
}

def map_and_anonymize(record):
    taxi_id = record[0]
    event_time = record[1]
    raw_lat = record[2]
    raw_lon = record[3]
    speed = record[4]
    status = record[5]

    city = "Casablanca"
    matched_zone_id = 99
    final_lat = raw_lat
    final_lon = raw_lon

    for zone_id, zone_data in CASABLANCA_ZONES.items():
        inside_lat = zone_data["min_lat"] <= raw_lat <= zone_data["max_lat"]
        inside_lon = zone_data["min_lon"] <= raw_lon <= zone_data["max_lon"]
        if inside_lat and inside_lon:
            matched_zone_id = zone_id
            final_lat = zone_data["centroid_lat"]
            final_lon = zone_data["centroid_lon"]
            break

    return (
        city,
        int(matched_zone_id),
        int(event_time),
        int(taxi_id),
        float(final_lat),
        float(final_lon),
        float(speed),
        str(status),
    )

anonymized_stream = valid_stream.map(
    map_and_anonymize,
    output_type=Types.TUPLE([
        Types.STRING(), Types.INT(), Types.LONG(), Types.INT(),
        Types.DOUBLE(), Types.DOUBLE(), Types.DOUBLE(), Types.STRING()
    ])
)

# Filter out zone_id 99 (out-of-scope GPS coordinates outside Casablanca zones)
filtered_anonymized_stream = anonymized_stream.filter(lambda record: record[1] != 99)

if DEBUG_STREAM_OUTPUT:
    filtered_anonymized_stream.print()
    print("Debug sink attached for filtered_anonymized_stream.")
else:
    print("Debug sink disabled for filtered_anonymized_stream.")

print("Map-matching, anonymization, and zone_id 99 filtering configured.")

Debug sink disabled for filtered_anonymized_stream.
Map-matching, anonymization, and zone_id 99 filtering configured.


### Step 4: Geospatial Anonymization via Zone Centroids - Privacy + Aggregation

**The Challenge**:
- Why can't we store raw GPS? **PRIVACY** → Individual movement tracking violates regulations (GDPR, local laws)
- Why can't we just drop GPS? **BUSINESS VALUE** → Need spatial data for demand analytics

**The Solution**: Zone Centroids
- Carve Casablanca into 16 zones (neighborhoods + commercial areas)
- Replace raw GPS with zone centroid (single point per zone)
- Result: Anonymized, privacy-respecting, still spatially meaningful

**Map-Matching Algorithm**:
```
For each valid GPS point (raw_lat, raw_lon, ...):
  1. Iterate through 16 zones (CASABLANCA_ZONES dict)
  2. Check: Is (raw_lat, raw_lon) inside zone's bounds?
     Inside bounds = (min_lat ≤ lat ≤ max_lat) AND (min_lon ≤ lon ≤ max_lon)
  3. If first match found:
     - Remember zone_id
     - Replace raw coords with zone's centroid
     - BREAK (stop searching, use first match)
  4. If no match (shouldn't happen after validation):
     - Assign zone_id=99 ("Unknown")
     - Keep raw coords (helps debugging)

Output Tuple: (city, zone_id, event_time, taxi_id, centroid_lat, centroid_lon, speed, status)
           Example: ("Casablanca", 1, 1700001234, 123, 33.60, -7.64, 42.5, "AVAILABLE")
```

**Privacy + Business Example**:
```
Raw: (33.605342, -7.641123)  ← Precise enough to track individual!

Zone 1 (Anfa):
  Bounds: lat [33.58, 33.62], lon [-7.66, -7.62]
  Centroid: (33.60, -7.64)

Output: (33.60, -7.64)  ← Precise to 10m, but not individual tracking!
```

**Result**: anonymized_stream ready for dual sinks (Kafka + Cassandra)

### Step 5: The Dual Sinks (Kafka & Cassandra) 🚰

**Objective:** Output the `anonymized_stream` to two distinct destinations.
1. **Kafka (`processed.gps`):** We must convert our Flink tuple back into a JSON string and publish it so Job 2 has a clean source to read from next week.
2. **Cassandra (`vehicle_positions`):** We will use Flink's Table API to write the stream directly into our NoSQL database, powering the live Grafana map.

### Why sink to Kafka processed.gps? (Cell 10)
The "Why": Think of your TaaSim architecture as a restaurant kitchen. Job 1 is the sous-chef who washes and chops the vegetables (normalizes the GPS). Job 2 (which you build next week) is the head chef who needs those chopped vegetables to make the final dish (calculating the demand ratio).

Job 2 cannot read from the raw GPS topic, because that data is full of errors, noise, and raw coordinates. Job 2 needs the clean, anonymized, zone-mapped data. So, Job 1 must put its finished work onto a new conveyor belt (the processed.gps topic)  so Job 2 can pick it up.

In [56]:
from pyflink.datastream.connectors.kafka import FlinkKafkaProducer
import time

def tuple_to_json(record):
    payload = {
        "schema_version": "1.0.0",
        "event_type": "GPS_NORMALIZED",
        "event_time": int(record[2]),
        "ingested_at": int(time.time()),
        "taxi_id": int(record[3]),
        "zone_id": int(record[1]),
        "centroid_lat": float(record[4]),
        "centroid_lon": float(record[5]),
        "speed_kmh": float(record[6]),
        "status": str(record[7]),
    }
    return json.dumps(payload)

json_stream = filtered_anonymized_stream.map(tuple_to_json, output_type=Types.STRING())
kafka_producer_props = {
    "bootstrap.servers": "localhost:9094",
    "transaction.timeout.ms": "900000",
}

kafka_sink = FlinkKafkaProducer(
    topic="processed.gps",
    serialization_schema=SimpleStringSchema(),
    producer_config=kafka_producer_props,
)

json_stream.add_sink(kafka_sink)
print("Kafka sink configured for processed.gps (zone_id 99 filtered out).")

Kafka sink configured for processed.gps (zone_id 99 filtered out).


### Step 5a: Kafka Sink (processed.gps) - Feeding the Next Job

**Why Dual Sinks?**
Same data needs different destinations for different purposes:
1. **Kafka** (processed.gps): Next job's INPUT (Job 2 reads this)
2. **Cassandra** (vehicle_positions): Dashboard's DATA (Grafana queries this)

**Kafka Sink Purpose**:
- Job 1 output becomes Job 2 input
- Job 2 will compute demand aggregation, trip matching, etc.
- If we didn't sink to Kafka, Job 2 would have to read raw.gps directly (which is noisy/unreliable)
- Separation of concerns: each job owns its output contract

**Transformation** (Tuple → JSON):
```
anonymized_stream (Tuple8):
  ("Casablanca", 1, 1700001234, 123, 33.60, -7.64, 42.5, "AVAILABLE")
        ↓ map(tuple_to_json)
  (JSON string):
  {
    "schema_version": "1.0.0",       ← Contract version (for compatibility)
    "event_type": "GPS_NORMALIZED",  ← Semantic label
    "event_time": 1700001234,        ← Original event time
    "ingested_at": 1776248493,       ← When Flink processed it
    "taxi_id": 123,
    "zone_id": 1,
    "centroid_lat": 33.60,
    "centroid_lon": -7.64,
    "speed_kmh": 42.5,
    "status": "AVAILABLE"
  }
```

**Why JSON?**
- Human-readable (easy debugging)
- Schema-flexible (Job 2 can ignore unknown fields)
- Version-proof (schema_version field allows evolution)

**Kafka Producer Config**:
- `bootstrap.servers`: localhost:9094 (same as consumer)
- `transaction.timeout.ms`: Extended time for batching (improves throughput)

### Why sink to Cassandra vehicle_positions? (Cell 11)
The "Why": Flink is terrible at answering user queries. If a passenger opens your mobile app and asks, "Where is the nearest taxi?", or if you open your Grafana dashboard to view the live map, Flink cannot answer that.

Cassandra is your Serving Layer. It is a lightning-fast NoSQL database designed to answer specific API queries instantly. Job 1's job is to constantly update Cassandra with the absolute latest position of every single taxi, so that when Grafana asks for the data every 10 seconds, the database has the answer ready

In [57]:
from pyflink.datastream.connectors.cassandra import CassandraSink

cassandra_query = (
    "INSERT INTO taasim.vehicle_positions "
    "(city, zone_id, event_time, taxi_id, latitude, longitude, speed, status) "
    "VALUES (?, ?, ?, ?, ?, ?, ?, ?);"
 )

cassandra_sink = (
    CassandraSink.add_sink(filtered_anonymized_stream)
    .set_host("localhost", 9042)
    .set_query(cassandra_query)
    .build()
 )

print("Cassandra sink configured for taasim.vehicle_positions (zone_id 99 filtered out).")
print("If running from containerized Flink, change host to cassandra.")

Cassandra sink configured for taasim.vehicle_positions (zone_id 99 filtered out).
If running from containerized Flink, change host to cassandra.


### Step 5b: Cassandra Sink (vehicle_positions) - Serving Dashboard Queries

**Why Cassandra for Dashboards?**
- **Real-time availability**: Cassandra distributed, fault-tolerant
- **Column-oriented**: Efficient for time-series queries (99% of dashboard queries are time-range scans)
- **No join latency**: Data replicated locally; Grafana queries return fast
- Unlike Kafka (append-only log), Cassandra allows efficient lookups by partition key

**Cassandra Table Schema** (vehicle_positions):
```sql
PRIMARY KEY (zone_id, event_time, taxi_id)
  ↓
Partition Key: zone_id     ← Data physically distributed by zone
Clustering Keys: event_time, taxi_id ← Sort order within partition
                                        MUST query with zone_id + optional time range
```

**Query Strategy** (how Grafana accesses):
```sql
-- EFFICIENT (partition key specified):
SELECT * FROM vehicle_positions 
WHERE zone_id = 1 AND event_time > 1700001000;  ✓ FAST

-- INEFFICIENT (missing partition key):
SELECT * FROM vehicle_positions 
WHERE event_time > 1700001000 ALLOW FILTERING;  ✗ SLOW (scans ALL partitions)
```

**Cassandra Sink Flow**:
```
anonymized_stream (Tuple8):
  ("Casablanca", 1, 1700001234, 123, 33.60, -7.64, 42.5, "AVAILABLE")
       ↓ CassandraSink
  INSERT query:
    INSERT INTO vehicle_positions 
      (zone_id, event_time, taxi_id, centroid_lat, centroid_lon, speed_kmh, status, city)
    VALUES (1, 1700001234, 123, 33.60, -7.64, 42.5, "AVAILABLE", "Casablanca")
```

**Host Binding**:
- `contact_points: 'localhost'` ← Cassandra running in Docker Compose
- Port 9042 ← Default Cassandra native protocol
- Keyspace: cassandra_database (created by docker-compose)
- Table: vehicle_positions (created by cassandra_schema.cql on startup)

In [58]:
# Final execution cell: submit once after all sinks are configured.
should_submit = True

if "job_client" in globals():
    try:
        current_status = job_client.get_job_status().result()
        print(f"Existing job status: {current_status}")
        print(f"Existing JobID: {job_client.get_job_id()}")

        if "RUNNING" in str(current_status):
            print("Job is already running. Skipping re-submit to avoid graph reset errors.")
            should_submit = False
        else:
            print("Existing job is not running. Submitting a fresh job...")
    except Exception as ex:
        print(f"Existing job client is stale/unavailable: {ex}")
        print("Submitting a fresh job...")

if should_submit:
    job_client = env.execute_async("Jupyter: Job 1 - Full GPS Normalizer Pipeline")
    print(f"Job submitted. JobID: {job_client.get_job_id()}")

Existing job status: JobStatus.RUNNING
Existing JobID: 9509ba3ababde326f0b329a40b1295c5
Job is already running. Skipping re-submit to avoid graph reset errors.


### Step 6: Job Submission & Async Execution - Why We Execute LAST

**Why Defer Execution to the End?**
Flink jobs are **lazy-evaluated**. Building a topology doesn't execute anything:
```
Build stream graph ≠ Execute stream graph
```

Example (build cost vs execution cost):
```python
# Build phase (cheap, no processing):
raw_stream = kafka_consumer(...)              # Just defines a connector
filtered = raw_stream.filter(...)             # Just defines a transformation
anonymized = filtered.map(...)                # Just defines a transformation, 2GB data flowing here!

# Execution phase (expensive, actual processing):
env.execute_async()  # ← NOW the data flows through ALL transformations above
```

**Why Async?**
- `env.execute()` (blocking): Notebook frozen until job finishes (hours/days)
- `env.execute_async()`: Job runs in background, notebook responsive, you can monitor/adjust

**Idempotent Submission Logic** (critical!):
```python
# Problem: If you re-run this cell, job submits AGAIN → duplicate data flow

# Solution: Only submit if no running job exists
if should_submit:  # First time only
    job_client = env.execute_async()
    print(f"Job submitted: {job_client.get_job_id()}")
else:  # Subsequent runs
    print(f"Job already running: {job_client.get_job_id()}")
    print(f"Status: {job_client.get_job_status()}")
```

**The `job_client` Object**:
- Returned by `env.execute_async()`
- Represents the running job in Flink mini-cluster
- Used to:
  - Get job_id (UUID unique per submission)
  - Check job_status (RUNNING, FINISHED, FAILED, etc.)
  - Cancel job if needed (`job_client.cancel()`)

**Data Flow Post-Submission**:
```
Job starts running in notebook's JVM:
  raw.gps → Kafka Consumer (pulls messages, port 9094)
         → structured tuple parsing
         → watermarking
         → validation filters
         → map-matching
         → anonymized tuple
         ├→ Kafka Sink (processed.gps, port 9094)
         └→ Cassandra Sink (vehicle_positions, port 9042)

All running concurrently inside one mini-cluster process!
Each transformation happens in parallel (partition-aware scaling)
```

**Next Steps After Execution**:
1. Monitor `job_client.get_job_status()` → should be RUNNING
2. Check Kafka `processed.gps` topic for output messages
3. Query Cassandra `vehicle_positions` for written records
4. Open Grafana dashboards to visualize data